In [3]:
import pandas as pd
import torch, torchvision, torchaudio
import torch.nn as nn 

from sklearn import model_selection, preprocessing
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from sklearn.metrics import mean_squared_error
torch.backends.cudnn.benchmark = True

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


### Recommender Systems 

1 – Matrix vectorization for recommender system

In [ ]:
# --- Load ratings ---
df = pd.read_csv('ml-32m/ratings.csv')

In [ ]:
# --- Quick peek ---
# Show the first 2 rows to verify the schema looks right.
df.head(2)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228


In [18]:
# --- Basic dataset stats ---
# Count distinct users and items to understand matrix shape.
print(f"Unique users: {df.userId.nunique()}, "
      f"Unique movies: {df.movieId.nunique()}")

Unique users: 200948, Unique movies: 84432


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB


In [19]:
# --- Data Class ---
# This custom Dataset class is used instead of plain NumPy arrays because it integrates 
# seamlessly with PyTorch's DataLoader, enabling efficient batching, shuffling, and parallel loading. 
# It also ensures samples are returned as torch.Tensors (ready for GPU training), allows on-the-fly 
# preprocessing via __getitem__, and scales better to very large datasets than raw arrays.
class MovieDataset(Dataset):
    def __init__(self, users, movies, ratings) -> None:
        """
        Custom PyTorch Dataset for user-movie-rating triplets.

        Parameters
        ----------
        users : array-like
            Sequence of user identifiers.
        movies : array-like
            Sequence of movie identifiers corresponding to the users.
        ratings : array-like
            Sequence of ratings given by the users to the movies.
        """
        super().__init__()
        self.users = users
        self.movies = movies
        self.ratings = ratings

    def __len__(self):
        """
        Returns
        -------
        int
            Total number of samples in the dataset.
        """
        return len(self.users)

    def __getitem__(self, idx):
        """
        Retrieve a single sample from the dataset.

        Parameters
        ----------
        idx : int
            Index of the desired sample.

        Returns
        -------
        tuple of torch.Tensor
            (user_id, movie_id, rating) as PyTorch tensors.
            - user_id: torch.long
            - movie_id: torch.long
            - rating: torch.long
        """
        users = self.users[idx]
        movies = self.movies[idx]
        ratings = self.ratings[idx]

        return (
            torch.tensor(users, dtype=torch.long),
            torch.tensor(movies, dtype=torch.long),
            torch.tensor(ratings, dtype=torch.long),
        )

In [21]:
# -----------------------------------------------------------------------------
# Overview: Embeddings & Rating Prediction (Recommender Context)
# -----------------------------------------------------------------------------
# In this model, each user ID and each movie ID is mapped to a learned dense
# vector (an embedding). These embeddings act as compact feature representations:
# - User embedding ≈ learned profile of user preferences
# - Movie embedding ≈ learned profile of movie characteristics
#
# During training, embeddings start random and are updated via backpropagation
# so that users with similar tastes and movies with similar audiences end up
# with similar vectors. To predict a rating for (user, movie), we:
#   1) look up the user embedding and the movie embedding,
#   2) combine them (here, by concatenation),
#   3) pass the combined vector through a linear layer to produce a single
#      scalar: the predicted rating.
#
# Notes:
# - EmbeddingBag is typically used when each example contains a "bag" (set)
#   of indices plus offsets; for single IDs per example, nn.Embedding is
#   often more straightforward.
# - The forward method name is misspelled below ("forwarad"); PyTorch will not
#   call it automatically unless named exactly `forward`.
# -----------------------------------------------------------------------------

# --- Model Class ---
class RecommenderSystemsModel(nn.Module):
    def __init__(self, n_users, n_movies, n_embeddings = 32):
        """
        Neural recommender with separate embedding tables for users and movies.
        
        Parameters
        ----------
        n_users : int
            Number of distinct users (size of the user ID vocabulary).
        n_movies : int
            Number of distinct movies (size of the movie ID vocabulary).
        n_embeddings : int, optional (default=32)
            Dimensionality of the embedding vectors for users and movies.
        """
        super().__init__()
        # Learnable table mapping each user ID -> embedding vector (size n_embeddings).
        # EmbeddingBag expects pooled lookups over bags; for single IDs per sample,
        # nn.Embedding is usually preferred.
        self.user_embed = nn.EmbeddingBag(n_users, n_embeddings)
        # Learnable table mapping each movie ID -> embedding vector (size n_embeddings).
        self.movie_embed = nn.EmbeddingBag(n_movies, n_embeddings)
        # Linear projection from concatenated [user_emb ; movie_emb] to a single rating.
        self.out = nn.Linear(n_embeddings * 2, 1)

    def forwarad(self, users, movies):
        """
        Compute predicted ratings for a batch of (user, movie) pairs.

        Parameters
        ----------
        users : torch.LongTensor
            Tensor of user IDs (indices into the user embedding table).
        movies : torch.LongTensor
            Tensor of movie IDs (indices into the movie embedding table).

        Returns
        -------
        torch.Tensor
            Predicted ratings with shape (batch_size, 1).
        """
        # Look up embeddings for users and movies.
        user_embeds = self.user_embed(users)
        movie_embeds = self.movie_embed(movies)
        # Concatenate embeddings along feature dimension to form joint representation.
        x = torch.cat([user_embeds, movie_embeds], dim=1)
        # Map the joint representation to a single scalar rating prediction.
        x = self.out(x)
        return x


In [32]:
# -------------------------------------------------------------------------
# Model training and evaluation setup
# -------------------------------------------------------------------------
# Before training, we must ensure that user IDs and movie IDs are encoded
# as contiguous integer indices starting from 0. This is required because
# PyTorch's nn.Embedding layers expect indices in the range [0, N-1],
# where N is the vocabulary size (number of distinct users or movies).
# -------------------------------------------------------------------------

# Encode user and movie IDs so they start at 0 and are contiguous.
lbl_user = preprocessing.LabelEncoder()
lbl_movie = preprocessing.LabelEncoder()

# Transform raw IDs into 0-based integer indices.
df['userId'] = lbl_user.fit_transform(df['userId'].values)
df['movieId'] = lbl_movie.fit_transform(df['movieId'].values)


In [ ]:
# -------------------------------------------------------------------------
# Train–test split
# -------------------------------------------------------------------------
# To evaluate the recommender system properly, we divide the dataset into
# a training set and a test set:
#   - Training set (80% of the data): used to fit the model parameters.
#   - Test set (20% of the data): held out and used only for evaluation,
#     ensuring that performance is measured on unseen examples.
#
# The parameter `random_state=123` fixes the random seed, so that the
# split is reproducible (same train/test partition every run).
# -------------------------------------------------------------------------

df_train, df_test = model_selection.train_test_split(
    df,
    test_size=0.2,
    random_state=123
)

In [ ]:
# -------------------------------------------------------------------------
# Dataset instances for training and validation
# -------------------------------------------------------------------------
# We now wrap the raw train/test splits into MovieDataset objects.
# This converts user IDs, movie IDs, and ratings into a PyTorch-compatible
# Dataset format, providing:
#   - __len__  → size of the dataset
#   - __getitem__ → returns (user_id, movie_id, rating) as torch.Tensors
#
# These dataset objects can then be passed to a DataLoader to enable
# efficient batching, shuffling, and GPU-compatible training.
# -------------------------------------------------------------------------

train_dataset = MovieDataset(
    users   = df_train.userId.values,
    movies  = df_train.movieId.values,
    ratings = df_train.rating.values
)

valid_dataset = MovieDataset(
    users   = df_test.userId.values,
    movies  = df_test.movieId.values,
    ratings = df_test.rating.values
)


,userId,movieId,rating,timestamp,userid
2249074,14294,13037,2.0,1288317970,14294
23619000,147966,449,4.0,845633368,147966
16506643,103420,18425,5.0,1591670054,103420
20648313,129283,21670,3.0,1620950345,129283
1857725,11766,2021,4.0,1283626892,11766
...,...,...,...,...,...
1241052,7974,332,4.0,838741024,7974
28329282,177584,15471,4.5,1316734339,177584
20999550,131545,52217,5.0,1624239938,131545
8666477,54324,3162,4.0,974649812,54324
